In [5]:
import pandas as pd
import numpy as np

# --------------------------------------------------------
# 1. INPUTS YOU ALREADY HAVE
# --------------------------------------------------------

# Hourly production rate (demand) for finished products
# These are from your table
product_hourly_rate = {
    "A1": 12.025,
    "A2": 24.0375,
    "A3": 31.25,
    "B1": 14.425,
    "B2": 19.225,
}

# Bill of Materials:
# bom[product][part] = qty of that part inside ONE unit of that product
# >>> TODO: replace all the 0/1/2/... values below with your actual Product-Parts matrix <<<
bom = {
    "A1": {"P1":1, "P2":4, "P3":0,  "P4":0, "P5":0,  "P6":1, "P7":0, "P8":0, "P9":0,
           "P10":1,"P11":2,"P12":4,"P13":2,"P14":0,"P15":0,"P16":0,"P17":2,"P18":4,"P19":0,"P20":0},
    "A2": {"P1":2, "P2":0, "P3":0,  "P4":1, "P5":1,  "P6":0, "P7":4, "P8":0, "P9":2,
           "P10":0,"P11":0,"P12":0,"P13":0,"P14":4,"P15":0,"P16":1,"P17":0,"P18":1,"P19":2,"P20":2},
    "A3": {"P1":4, "P2":2, "P3":0,  "P4":2, "P5":2,  "P6":0, "P7":0, "P8":0, "P9":2,
           "P10":1,"P11":0,"P12":0,"P13":0,"P14":2,"P15":0,"P16":4,"P17":0,"P18":0,"P19":4,"P20":3},
    "B1": {"P1":4, "P2":2, "P3":4,  "P4":2, "P5":0,  "P6":0, "P7":0, "P8":4, "P9":0,
           "P10":1,"P11":0,"P12":0,"P13":0,"P14":2,"P15":2,"P16":0,"P17":0,"P18":0,"P19":4,"P20":3},
    "B2": {"P1":1, "P2":0, "P3":1,  "P4":0, "P5":0,  "P6":2, "P7":4, "P8":0, "P9":0,
           "P10":0,"P11":4,"P12":4,"P13":2,"P14":2,"P15":0,"P16":0,"P17":2,"P18":4,"P19":1,"P20":0},
}

# Part physical dimensions in inches (from the case table)
# P#: (X_in, Y_in, Z_in)
# >>> keep these consistent with the PDF table <<<
part_dims = {
    "P1":  (2, 6, 6),
    "P2":  (8, 8, 4),
    "P3":  (6, 6, 6),
    "P4":  (12, 6, 4),
    "P5":  (8, 4, 6),
    "P6":  (2, 8, 6),
    "P7":  (2, 2, 12),
    "P8":  (4, 4, 4),
    "P9":  (2, 4, 12),
    "P10": (4, 4, 4),
    "P11": (4, 6, 4),
    "P12": (6, 6, 4),
    "P13": (2, 2, 12),
    "P14": (2, 4, 6),
    "P15": (4, 6, 4),
    "P16": (4, 4, 4),
    "P17": (12, 2, 2),
    "P18": (12, 2, 2),
    "P19": (12, 2, 2),
    "P20": (12, 2, 2),
}

# Planning constants
A_AUTONOMY_HOURS = 4    # Client A wants 4h autonomy @99%
B_AUTONOMY_HOURS = 12   # Client B wants 12h autonomy @99%
# We will start with deterministic planning (SafetyFactor = 1)

# Allocation policy:
# - We keep at least 1 hour of A's demand at Client A's warehouse
# - We keep at least 4 hours of B's demand at Client B's warehouse
A_SITE_HOURS = 1
B_SITE_HOURS = 4

# Space + cost assumptions
STACK_HEIGHT_FT = 20.0        # usable vertical height at near-client warehouses
COST_WAREHOUSE_FT2 = 200.0    # $/ft² near client
COST_FACTORY_FT2   = 250.0    # $/ft² at factory outbound (later used in Task 3 comparisons)


# --------------------------------------------------------
# 2. BUILD A DATAFRAME OF HOURLY USAGE PER PART
# --------------------------------------------------------

products_A = ["A1", "A2", "A3"]
products_B = ["B1", "B2"]

# Get list of all parts from BOM keys
parts = sorted({part for prod in bom for part in bom[prod].keys()})

rows = []
for part in parts:
    # Hourly demand of this part from Client A products
    hourly_A = 0.0
    for prod in products_A:
        qty_per_unit = bom[prod].get(part, 0)
        prod_rate = product_hourly_rate[prod]  # units/hour of that finished product
        hourly_A += qty_per_unit * prod_rate

    # Hourly demand of this part from Client B products
    hourly_B = 0.0
    for prod in products_B:
        qty_per_unit = bom[prod].get(part, 0)
        prod_rate = product_hourly_rate[prod]
        hourly_B += qty_per_unit * prod_rate

    rows.append({
        "part": part,
        "A_hourly_demand": hourly_A,  # parts/hour consumed by Client A
        "B_hourly_demand": hourly_B,  # parts/hour consumed by Client B
    })

df_parts = pd.DataFrame(rows).set_index("part")


# --------------------------------------------------------
# 3. COMPUTE BUFFER REQUIREMENTS
#    A needs 4 hours worth, B needs 12 hours worth
#    (baseline: no extra variability factor yet)
# --------------------------------------------------------

df_parts["A_buffer_units"] = df_parts["A_hourly_demand"] * A_AUTONOMY_HOURS
df_parts["B_buffer_units"] = df_parts["B_hourly_demand"] * B_AUTONOMY_HOURS


# --------------------------------------------------------
# 4. SPLIT BUFFER BETWEEN FACTORY OUTBOUND VS. NEAR-CLIENT WAREHOUSES
# --------------------------------------------------------

# minimum stock that must live physically at each client's site (local buffer before next truck)
df_parts["A_site_units_min"] = df_parts["A_hourly_demand"] * A_SITE_HOURS
df_parts["B_site_units_min"] = df_parts["B_hourly_demand"] * B_SITE_HOURS

# can't exceed total buffer
df_parts["WH_A_units"] = np.minimum(df_parts["A_site_units_min"], df_parts["A_buffer_units"])
df_parts["WH_B_units"] = np.minimum(df_parts["B_site_units_min"], df_parts["B_buffer_units"])

# whatever is left after placing local site stock sits at factory outbound
df_parts["Factory_units_from_A"] = df_parts["A_buffer_units"] - df_parts["WH_A_units"]
df_parts["Factory_units_from_B"] = df_parts["B_buffer_units"] - df_parts["WH_B_units"]

df_parts["Factory_units"] = df_parts["Factory_units_from_A"] + df_parts["Factory_units_from_B"]


# --------------------------------------------------------
# 5. SPACE, AREA, COST
#    Convert units -> volume -> floor area -> $
# --------------------------------------------------------

def unit_vol_ft3(part_id):
    x_in, y_in, z_in = part_dims[part_id]
    x_ft = x_in / 12.0
    y_ft = y_in / 12.0
    z_ft = z_in / 12.0
    return x_ft * y_ft * z_ft

df_parts["unit_vol_ft3"] = df_parts.index.map(unit_vol_ft3)

# volume at each location (cubic ft)
df_parts["vol_factory_ft3"] = df_parts["Factory_units"] * df_parts["unit_vol_ft3"]
df_parts["vol_A_ft3"]       = df_parts["WH_A_units"]   * df_parts["unit_vol_ft3"]
df_parts["vol_B_ft3"]       = df_parts["WH_B_units"]   * df_parts["unit_vol_ft3"]

# convert to floor area in ft² assuming 20 ft stacking height
df_parts["area_factory_ft2"] = df_parts["vol_factory_ft3"] / STACK_HEIGHT_FT
df_parts["area_A_ft2"]       = df_parts["vol_A_ft3"]       / STACK_HEIGHT_FT
df_parts["area_B_ft2"]       = df_parts["vol_B_ft3"]       / STACK_HEIGHT_FT

# cost impact
df_parts["cost_factory_$"] = df_parts["area_factory_ft2"] * COST_FACTORY_FT2
df_parts["cost_A_$"]       = df_parts["area_A_ft2"]       * COST_WAREHOUSE_FT2
df_parts["cost_B_$"]       = df_parts["area_B_ft2"]       * COST_WAREHOUSE_FT2


# --------------------------------------------------------
# 6. SUMMARY TOTALS (useful for Task 2 writeup)
# --------------------------------------------------------

summary = {
    "Total_factory_area_ft2": df_parts["area_factory_ft2"].sum(),
    "Total_A_area_ft2":       df_parts["area_A_ft2"].sum(),
    "Total_B_area_ft2":       df_parts["area_B_ft2"].sum(),
    "Total_factory_cost_$":   df_parts["cost_factory_$"].sum(),
    "Total_A_cost_$":         df_parts["cost_A_$"].sum(),
    "Total_B_cost_$":         df_parts["cost_B_$"].sum(),
}

summary_df = pd.DataFrame([summary])

# --------------------------------------------------------
# 7. DISPLAY MAIN OUTPUTS
# --------------------------------------------------------

print("=== PER-PART STORAGE PLAN (Task 2 core table) ===")
print(
    df_parts[[
        "A_hourly_demand","B_hourly_demand",
        "A_buffer_units","B_buffer_units",
        "Factory_units","WH_A_units","WH_B_units",
        "area_factory_ft2","area_A_ft2","area_B_ft2",
        "cost_factory_$","cost_A_$","cost_B_$"
    ]]
)

print("\n=== TOTAL SPACE/COST SUMMARY ===")
print(summary_df)


=== PER-PART STORAGE PLAN (Task 2 core table) ===
      A_hourly_demand  B_hourly_demand  A_buffer_units  B_buffer_units  \
part                                                                     
P1           185.1000           76.925          740.40           923.1   
P10           43.2750           14.425          173.10           173.1   
P11           24.0500           76.900           96.20           922.8   
P12           48.1000           76.900          192.40           922.8   
P13           24.0500           38.450           96.20           461.4   
P14          158.6500           67.300          634.60           807.6   
P15            0.0000           28.850            0.00           346.2   
P16          149.0375            0.000          596.15             0.0   
P17           24.0500           38.450           96.20           461.4   
P18           72.1375           76.900          288.55           922.8   
P19          173.0750           76.925          692.30        

In [7]:
import pandas as pd
import numpy as np

# --------------------------------------------------------
# 1. INPUTS
# --------------------------------------------------------

product_hourly_rate = {
    "A1": 12.025,
    "A2": 24.0375,
    "A3": 31.25,
    "B1": 14.425,
    "B2": 19.225,
}

bom = {
    "A1": {"P1":1, "P2":4, "P3":0,  "P4":0, "P5":0,  "P6":1, "P7":0, "P8":0, "P9":0,
           "P10":1,"P11":2,"P12":4,"P13":2,"P14":0,"P15":0,"P16":0,"P17":2,"P18":4,"P19":0,"P20":0},
    "A2": {"P1":2, "P2":0, "P3":0,  "P4":1, "P5":1,  "P6":0, "P7":4, "P8":0, "P9":2,
           "P10":0,"P11":0,"P12":0,"P13":0,"P14":4,"P15":0,"P16":1,"P17":0,"P18":1,"P19":2,"P20":2},
    "A3": {"P1":4, "P2":2, "P3":0,  "P4":2, "P5":2,  "P6":0, "P7":0, "P8":0, "P9":2,
           "P10":1,"P11":0,"P12":0,"P13":0,"P14":2,"P15":0,"P16":4,"P17":0,"P18":0,"P19":4,"P20":3},
    "B1": {"P1":4, "P2":2, "P3":4,  "P4":2, "P5":0,  "P6":0, "P7":0, "P8":4, "P9":0,
           "P10":1,"P11":0,"P12":0,"P13":0,"P14":2,"P15":2,"P16":0,"P17":0,"P18":0,"P19":4,"P20":3},
    "B2": {"P1":1, "P2":0, "P3":1,  "P4":0, "P5":0,  "P6":2, "P7":4, "P8":0, "P9":0,
           "P10":0,"P11":4,"P12":4,"P13":2,"P14":2,"P15":0,"P16":0,"P17":2,"P18":4,"P19":1,"P20":0},
}

part_dims = {
    "P1":  (2, 6, 6), "P2":  (8, 8, 4), "P3":  (6, 6, 6), "P4":  (12, 6, 4),
    "P5":  (8, 4, 6), "P6":  (2, 8, 6), "P7":  (2, 2, 12), "P8":  (4, 4, 4),
    "P9":  (2, 4, 12), "P10": (4, 4, 4), "P11": (4, 6, 4), "P12": (6, 6, 4),
    "P13": (2, 2, 12), "P14": (2, 4, 6), "P15": (4, 6, 4), "P16": (4, 4, 4),
    "P17": (12, 2, 2), "P18": (12, 2, 2), "P19": (12, 2, 2), "P20": (12, 2, 2)
}

A_AUTONOMY_HOURS = 4
B_AUTONOMY_HOURS = 12
A_SITE_HOURS = 1
B_SITE_HOURS = 4
STACK_HEIGHT_FT = 20.0
COST_WAREHOUSE_FT2 = 200.0
COST_FACTORY_FT2 = 250.0

# --------------------------------------------------------
# 2. COMPUTE HOURLY PART DEMAND
# --------------------------------------------------------

products_A = ["A1", "A2", "A3"]
products_B = ["B1", "B2"]
parts = sorted({part for prod in bom for part in bom[prod].keys()})

rows = []
for part in parts:
    hourly_A = sum(bom[p].get(part, 0) * product_hourly_rate[p] for p in products_A)
    hourly_B = sum(bom[p].get(part, 0) * product_hourly_rate[p] for p in products_B)
    rows.append({"part": part, "A_hourly_demand": hourly_A, "B_hourly_demand": hourly_B})

df_parts = pd.DataFrame(rows).set_index("part")

# --------------------------------------------------------
# 3. BUFFER CALCULATION
# --------------------------------------------------------

df_parts["A_buffer_units"] = df_parts["A_hourly_demand"] * A_AUTONOMY_HOURS
df_parts["B_buffer_units"] = df_parts["B_hourly_demand"] * B_AUTONOMY_HOURS

df_parts["A_site_units_min"] = df_parts["A_hourly_demand"] * A_SITE_HOURS
df_parts["B_site_units_min"] = df_parts["B_hourly_demand"] * B_SITE_HOURS

df_parts["WH_A_units"] = np.minimum(df_parts["A_site_units_min"], df_parts["A_buffer_units"])
df_parts["WH_B_units"] = np.minimum(df_parts["B_site_units_min"], df_parts["B_buffer_units"])

df_parts["Factory_units_from_A"] = df_parts["A_buffer_units"] - df_parts["WH_A_units"]
df_parts["Factory_units_from_B"] = df_parts["B_buffer_units"] - df_parts["WH_B_units"]
df_parts["Factory_units"] = df_parts["Factory_units_from_A"] + df_parts["Factory_units_from_B"]

# --------------------------------------------------------
# 4. SPACE, AREA, COST
# --------------------------------------------------------

def unit_vol_ft3(part_id):
    x_in, y_in, z_in = part_dims[part_id]
    return (x_in / 12) * (y_in / 12) * (z_in / 12)

df_parts["unit_vol_ft3"] = df_parts.index.map(unit_vol_ft3)

df_parts["vol_factory_ft3"] = df_parts["Factory_units"] * df_parts["unit_vol_ft3"]
df_parts["vol_A_ft3"] = df_parts["WH_A_units"] * df_parts["unit_vol_ft3"]
df_parts["vol_B_ft3"] = df_parts["WH_B_units"] * df_parts["unit_vol_ft3"]

df_parts["area_factory_ft2"] = df_parts["vol_factory_ft3"] / STACK_HEIGHT_FT
df_parts["area_A_ft2"] = df_parts["vol_A_ft3"] / STACK_HEIGHT_FT
df_parts["area_B_ft2"] = df_parts["vol_B_ft3"] / STACK_HEIGHT_FT

df_parts["cost_factory_$"] = df_parts["area_factory_ft2"] * COST_FACTORY_FT2
df_parts["cost_A_$"] = df_parts["area_A_ft2"] * COST_WAREHOUSE_FT2
df_parts["cost_B_$"] = df_parts["area_B_ft2"] * COST_WAREHOUSE_FT2

# --------------------------------------------------------
# 5. SUMMARY TOTALS
# --------------------------------------------------------

summary = {
    "Total_factory_area_ft2": df_parts["area_factory_ft2"].sum(),
    "Total_A_area_ft2": df_parts["area_A_ft2"].sum(),
    "Total_B_area_ft2": df_parts["area_B_ft2"].sum(),
    "Total_factory_cost_$": df_parts["cost_factory_$"].sum(),
    "Total_A_cost_$": df_parts["cost_A_$"].sum(),
    "Total_B_cost_$": df_parts["cost_B_$"].sum(),
}
summary_df = pd.DataFrame([summary])

# --------------------------------------------------------
# 6. EXPORT BOTH TABLES INTO ONE EXCEL SHEET
# --------------------------------------------------------

output_path = "FaMoaSa_Task2_StoragePlan.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_parts.to_excel(writer, sheet_name="Task2_Output")
    start_row = len(df_parts) + 3  # leave 2 blank lines
    summary_df.to_excel(writer, sheet_name="Task2_Output", startrow=start_row, index=False)

print(f"✅ File saved successfully: {output_path}")


✅ File saved successfully: FaMoaSa_Task2_StoragePlan.xlsx
